<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_07_model_evaluation/stage_07_model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07 – Model Evaluation**


# **Preparación de entorno**

## **1. Imports**

In [26]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-12 18:08:17,059 | INFO | Environment initialized


## **2. Acceso a drive**

In [27]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

2026-04-12 18:08:19,323 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **3. Carga de métricas**

In [28]:
from pathlib import Path
import pandas as pd

def load_all_classification_metrics(
    *,
    base_dir: Path,
    pattern: str = "classification_*_metrics.parquet",
    verbose: bool = True,
) -> dict[str, pd.DataFrame]:
    """
    Carga todos los archivos parquet de métricas de clasificación.

    Retorna:
        dict {model_name: DataFrame}
    """

    paths = list(base_dir.glob(pattern))
    results = {}

    if verbose:
        print(f"[INFO] Encontrados {len(paths)} archivos")

    for path in paths:
        name = path.stem.replace("classification_", "").replace("_metrics", "")

        if verbose:
            print(f"[LOAD] {name} -> {path}")

        df = pd.read_parquet(path)
        results[name] = df

    return results

In [29]:
def concat_metrics(metrics_dict: dict[str, pd.DataFrame]) -> pd.DataFrame:
    df_all = []

    for name, df in metrics_dict.items():
        df_copy = df.copy()
        df_copy["model"] = name
        df_all.append(df_copy)

    return pd.concat(df_all, ignore_index=True)

In [30]:
metrics_dir = DRIVE_DIR / "metrics/classification_metrics"

metrics_dict = load_all_classification_metrics(base_dir=metrics_dir)

# Ejemplo:
#metrics_dict["gru"].head()
#metrics_dict["lightgbm_balanced"].head()

df_all = pd.concat(
    [df.assign(model=name) for name, df in metrics_dict.items()],
    ignore_index=True
)

[INFO] Encontrados 14 archivos
[LOAD] logistic_regression -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_logistic_regression_metrics.parquet
[LOAD] logistic_regression_balanced -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_logistic_regression_balanced_metrics.parquet
[LOAD] random_forest -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_random_forest_metrics.parquet
[LOAD] random_forest_balanced -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_random_forest_balanced_metrics.parquet
[LOAD] xgboost -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_xgboost_metrics.parquet
[LOAD] xgboost_balanced -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_xgboost_balanced_metrics.parquet
[LOAD] lightgbm -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classi

# PLAN DE SELECCIÓN Y TUNING (SEQ2ONE T2)


## **7. Análisis conjunto (modelo + L + target)**

**Objetivo:** Encontrar configuraciones óptimas reales.

* Ranking por combinación:

  * model + window_size + target
* Top N configuraciones

👉 Resultado:

* mejores setups concretos

---

## **8. Selección final de candidatos**

**Objetivo:** Reducir el espacio de búsqueda.

Elegir:

### ✔ 3 modelos

* 1 deep learning (ej: transformer)
* 1 ensemble (rf / xgb)
* 1 baseline (logistic)

### ✔ 2–3 ventanas

* las mejores según análisis

### ✔ 1–2 targets

* los más estables

---

## **9. Definición del espacio de tuning**

**Objetivo:** Preparar optimización eficiente.

Para cada modelo:

* definir hiperparámetros clave
* limitar rango (no búsqueda infinita)

Ejemplo:

* RF → depth, n_estimators
* XGB → learning_rate, max_depth
* Transformer → d_model, heads, dropout

---

## **10. Ejecución de tuning**

**Objetivo:** Optimizar performance real.

* usar:

  * valid para tuning
  * test solo para evaluación final
* mantener:

  * mismo pipeline
  * mismo split

---

## **11. Evaluación final**

**Objetivo:** Validar modelo ganador.

* comparar:

  * vs naive
  * vs baseline
* revisar:

  * estabilidad entre splits
  * consistencia por régimen (opcional)

---

## **12. Conclusión y cierre del stage**

**Objetivo:** Formalizar resultados.

* modelo ganador
* configuración óptima
* interpretación:

  * qué aprendió el modelo
  * qué señal existe en los datos

---

# 🔹 RESUMEN CLAVE

El flujo es:

```
Resultados → Filtrado → Ranking → Selección → Tuning → Validación
```

---

Si quieres, en el siguiente paso ejecutamos juntos el **punto 4 (ranking real de tus modelos)** y te doy directamente el shortlist óptimo.


# **1. Consolidación de resultados**

**Objetivo:** Tener una base única para análisis.

* Unificar todos los parquets en un solo DataFrame (`df_all`)
* Verificar columnas, tipos y consistencia
* Confirmar:

  * splits: train / valid / test
  * targets: 90 / 120
  * ventanas: L ∈ {30, 60, 90, 120, 180}

In [31]:
df_all

,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,recall_macro,balanced_accuracy_naive,bal_acc_gain_vs_naive,horizon_min,class_weight_mode,family
0,logistic_regression,test,30,t2_dir_thr_120,93990,0.345316,0.270370,0.438369,0.579987,0.383358,0.345316,0.333333,0.011983,120,none,logistic_regression
1,logistic_regression,valid,30,t2_dir_thr_120,93508,0.335323,0.283183,0.604690,0.720698,0.551519,0.335323,0.333333,0.001990,120,none,logistic_regression
2,logistic_regression,test,30,t2_dir_thr_90,93990,0.343799,0.268514,0.436526,0.577668,0.409567,0.343799,0.333333,0.010466,90,none,logistic_regression
3,logistic_regression,valid,30,t2_dir_thr_90,93508,0.335520,0.282303,0.596703,0.714420,0.497953,0.335520,0.333333,0.002187,90,none,logistic_regression
4,logistic_regression,test,60,t2_dir_thr_120,88140,0.344856,0.265729,0.420574,0.564772,0.428477,0.344856,0.333333,0.011523,120,none,logistic_regression
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275,transformer_balanced,valid,120,t2_dir_thr_90,76048,0.416400,0.412012,0.596041,0.617229,0.434474,0.416400,0.333333,0.083067,90,balanced,transformer_balanced
276,transformer_balanced,test,180,t2_dir_thr_120,64740,0.445395,0.445841,0.514053,0.519741,0.447254,0.445395,0.333333,0.112062,120,balanced,transformer_balanced
277,transformer_balanced,valid,180,t2_dir_thr_120,64408,0.409654,0.410077,0.603517,0.618681,0.420697,0.409654,0.333333,0.076321,120,balanced,transformer_balanced
278,transformer_balanced,test,180,t2_dir_thr_90,64740,0.429520,0.425170,0.484447,0.492678,0.427173,0.429520,0.333333,0.096187,90,balanced,transformer_balanced


## 1.1. Verificación estructural

In [32]:
print('1. Validación estructural:\n')
df_all.info()
df_all.head()

1. Validación estructural:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   model                    280 non-null    object 
 1   split                    280 non-null    object 
 2   window_size              280 non-null    int64  
 3   target                   280 non-null    object 
 4   n_samples                280 non-null    int64  
 5   balanced_accuracy        280 non-null    float64
 6   f1_macro                 280 non-null    float64
 7   f1_weighted              280 non-null    float64
 8   accuracy                 280 non-null    float64
 9   precision_macro          280 non-null    float64
 10  recall_macro             280 non-null    float64
 11  balanced_accuracy_naive  280 non-null    float64
 12  bal_acc_gain_vs_naive    280 non-null    float64
 13  horizon_min              280 non-null    int64  
 14

,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,recall_macro,balanced_accuracy_naive,bal_acc_gain_vs_naive,horizon_min,class_weight_mode,family
0,logistic_regression,test,30,t2_dir_thr_120,93990,0.345316,0.270370,0.438369,0.579987,0.383358,0.345316,0.333333,0.011983,120,none,logistic_regression
1,logistic_regression,valid,30,t2_dir_thr_120,93508,0.335323,0.283183,0.604690,0.720698,0.551519,0.335323,0.333333,0.001990,120,none,logistic_regression
2,logistic_regression,test,30,t2_dir_thr_90,93990,0.343799,0.268514,0.436526,0.577668,0.409567,0.343799,0.333333,0.010466,90,none,logistic_regression
3,logistic_regression,valid,30,t2_dir_thr_90,93508,0.335520,0.282303,0.596703,0.714420,0.497953,0.335520,0.333333,0.002187,90,none,logistic_regression
4,logistic_regression,test,60,t2_dir_thr_120,88140,0.344856,0.265729,0.420574,0.564772,0.428477,0.344856,0.333333,0.011523,120,none,logistic_regression


## 1.2. Verificación columnas clave

In [33]:
print('2. Verificación columnas clave:\n')


cols_expected = [
    "model", "split", "window_size", "target",
    "balanced_accuracy", "bal_acc_gain_vs_naive",
    "horizon_min", "class_weight_mode"
]

missing = [c for c in cols_expected if c not in df_all.columns]
print("Missing:", missing)

2. Verificación columnas clave:

Missing: []


## 1.3. Validar splits

In [34]:
print('3. Verificación splits:\n')
df_all["split"].value_counts()

3. Verificación splits:



,count
split,
test,140
valid,140


## 1.4. Validar targets

In [35]:
print('4. Validar targets:\n')
df_all["target"].value_counts()

4. Validar targets:



,count
target,
t2_dir_thr_120,140
t2_dir_thr_90,140


## 1.5. Validar ventanas (window_size)

In [36]:
print('5. Validar ventanas:\n')

sorted(df_all["window_size"].unique())

5. Validar ventanas:



[np.int64(30), np.int64(60), np.int64(90), np.int64(120), np.int64(180)]

## 1.6. Validar duplicados

In [37]:
print('6. Validar duplicados:\n')

dup_cols = ["model", "split", "window_size", "target"]
df_all.duplicated(subset=dup_cols).sum()

6. Validar duplicados:



np.int64(0)

## 1.7. Validar NaNs

In [38]:
print('7. Validar NaNs:\n')
df_all.isna().sum()

7. Validar NaNs:



,0
model,0
split,0
window_size,0
target,0
n_samples,0
balanced_accuracy,0
f1_macro,0
f1_weighted,0
accuracy,0
precision_macro,0


## 1.8. Validar baseline

In [39]:
print('8. Validar Baseline:\n')
df_all["balanced_accuracy_naive"].unique()

8. Validar Baseline:



array([0.33333333])

## 1.9. Validar consistencia gain

In [40]:
print('9. Validar consistencia gain:\n')
(df_all["balanced_accuracy"] - df_all["balanced_accuracy_naive"]
 - df_all["bal_acc_gain_vs_naive"]).abs().max()

9. Validar consistencia gain:



0.0

## 1.10. Validación final rápida

In [41]:
print('10. Validación final rápida:\n')

print("Shape:", df_all.shape)
print("Models:", df_all["model"].nunique())
print("Configs:", df_all.groupby(["model","window_size","target"]).ngroups)

10. Validación final rápida:

Shape: (280, 16)
Models: 14
Configs: 140


# **2. Definición de métricas de decisión**

**Objetivo:** Establecer criterio correcto (evitar conclusiones erróneas).

* Métrica principal:
  → `bal_acc_gain_vs_naive`

* Métricas de apoyo:
  → `balanced_accuracy`, `f1_macro`

* Regla:

  * modelo útil → gain > 0
  * modelo fuerte → gain > 0.03–0.05

## 2.1. Creación de flags de calidad

Qué representa cada flag:

- is_useful: supera al naive
- is_strong: señal ya interesante
- is_very_strong: señal claramente fuerte

In [42]:
df = df_all.copy()

# limpiar class_weight
df["class_weight_mode"] = df["class_weight_mode"].fillna("none")

# flags
df["is_useful"] = df["bal_acc_gain_vs_naive"] > 0
df["is_strong"] = df["bal_acc_gain_vs_naive"] > 0.03
df["is_very_strong"] = df["bal_acc_gain_vs_naive"] > 0.05

In [43]:
df[[
    "model", "split", "window_size", "target",
    "bal_acc_gain_vs_naive", "is_useful", "is_strong", "is_very_strong"
]].head()

,model,split,window_size,target,bal_acc_gain_vs_naive,is_useful,is_strong,is_very_strong
0,logistic_regression,test,30,t2_dir_thr_120,0.011983,True,False,False
1,logistic_regression,valid,30,t2_dir_thr_120,0.001990,True,False,False
2,logistic_regression,test,30,t2_dir_thr_90,0.010466,True,False,False
3,logistic_regression,valid,30,t2_dir_thr_90,0.002187,True,False,False
4,logistic_regression,test,60,t2_dir_thr_120,0.011523,True,False,False


## 2.2. Revisar distribución de señal

In [44]:
df.groupby("split")["bal_acc_gain_vs_naive"].describe()

,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,140.0,0.046928,0.039747,0.006511,0.012122,0.017595,0.088136,0.115995
valid,140.0,0.035929,0.037644,0.000359,0.003245,0.007586,0.074878,0.103347


a) Diagnóstico de la distribución de señal

- Datos para el split test:

  * mean: 0.0469
  * median: 0.0176
  * percentil 75: 0.0881
  * máximo: 0.116

- Datos para el split valid:

  * mean: 0.0359
  * median: 0.0076
  * percentil 75: 0.0749
  * máximo: 0.103

---

b) Interpretación de la señal

1. Existe señal real en el dataset

    * El valor medio del gain es mayor que 0 tanto en test como en valid
    * Esto indica que los modelos superan al baseline naive
    * Por lo tanto, el dataset contiene alpha

2. La señal es altamente desigual

    * La mediana en test (0.0176) es baja
    * La media en test (0.0469) es significativamente mayor
    * Esto implica que la distribución está sesgada por pocos valores altos
    * Conclusión:

      * la mayoría de configuraciones tienen señal débil o nula
      * un subconjunto pequeño tiene señal fuerte

3. Existen configuraciones con señal muy fuerte

    * El máximo en test es aproximadamente 0.11
    * Este nivel de gain es alto para un problema T2 multiclase
    * Indica que ciertos modelos/configuraciones capturan bien la señal

4. Diferencia entre test y valid

    * mean test: 0.0469
    * mean valid: 0.0359
    * Existe una caída leve al pasar de test a valid
    * Interpretación:

      * hay algo de sobreajuste, pero no es severo
      * la señal se mantiene fuera de muestra

---

c) Conclusión técnica

El comportamiento observado corresponde a un caso de “sparse alpha”:

* la señal no está distribuida de forma uniforme
* depende de combinaciones específicas de:

  * modelo
  * window_size
  * target

---

d) Implicaciones para el análisis

* No es adecuado promediar resultados globalmente
* No es suficiente evaluar modelos por promedio general
* Es necesario identificar configuraciones específicas con alto rendimiento

---

e) Criterio de selección a partir de este punto

Se deben priorizar configuraciones con:

* gain > 0.05 → señal fuerte
* gain > 0.08 → señal muy fuerte

---

f) Estado del proceso

* El análisis de distribución de señal está completado
* Se confirma que:

  * existe señal
  * no es uniforme
  * hay configuraciones claramente superiores


# **3. Filtro de calidad (anti-ruido)**


Objetivo: eliminar configuraciones no robustas y quedarnos solo con señal consistente.

a) Selección inicial

* Trabajar principalmente con `split = test`
* Aplicar un umbral mínimo de señal:

  * `bal_acc_gain_vs_naive > 0.01` (filtra ruido puro)

b) Identificación de señal relevante

* Marcar como candidatos:

  * gain > 0.03 → señal moderada
  * gain > 0.05 → señal fuerte

c) Validación de consistencia (clave)

* Comparar cada configuración en `test` contra `valid`
* Mantener solo configuraciones donde:

  * `gain_valid > 0`
* Opcional (más estricto):

  * evitar casos donde `test ≫ valid`

d) Eliminación de falsos positivos

* Descartar configuraciones donde:

  * buen resultado en test pero nulo o muy bajo en valid
* Esto reduce riesgo de overfitting

e) Resultado esperado

* Dataset reducido con:

  * configuraciones con señal real
  * consistentes entre valid y test
* Base confiable para ranking y selección final




In [45]:
# ================================
# 3. Filtro de calidad (anti-ruido)
# ================================

df_filt = df.copy()

# a) Separar splits
df_test = df_filt[df_filt["split"] == "test"].copy()
df_valid = df_filt[df_filt["split"] == "valid"].copy()

# b) Renombrar métricas para merge
df_test = df_test.rename(columns={
    "balanced_accuracy": "balanced_accuracy_test",
    "f1_macro": "f1_macro_test",
    "bal_acc_gain_vs_naive": "bal_acc_gain_vs_naive_test",
})

df_valid = df_valid.rename(columns={
    "balanced_accuracy": "balanced_accuracy_valid",
    "f1_macro": "f1_macro_valid",
    "bal_acc_gain_vs_naive": "bal_acc_gain_vs_naive_valid",
})

# c) Merge por configuración
merge_keys = ["model", "window_size", "target"]

df_filtered = df_test.merge(
    df_valid[
        merge_keys + [
            "balanced_accuracy_valid",
            "f1_macro_valid",
            "bal_acc_gain_vs_naive_valid",
        ]
    ],
    on=merge_keys,
    how="inner"
)

# d) Filtro anti-ruido
# - test con gain > 0.01
# - valid con gain > 0
df_filtered = df_filtered[
    (df_filtered["bal_acc_gain_vs_naive_test"] > 0.01) &
    (df_filtered["bal_acc_gain_vs_naive_valid"] > 0.00)
].copy()

# e) Gap entre test y valid para inspección
df_filtered["gain_gap_test_valid"] = (
    df_filtered["bal_acc_gain_vs_naive_test"] -
    df_filtered["bal_acc_gain_vs_naive_valid"]
)

# f) Flags opcionales de robustez
df_filtered["is_consistent"] = df_filtered["gain_gap_test_valid"] <= 0.03
df_filtered["is_very_consistent"] = df_filtered["gain_gap_test_valid"] <= 0.02

# g) Ordenar por mejor señal en test
df_filtered = df_filtered.sort_values(
    by="bal_acc_gain_vs_naive_test",
    ascending=False
).reset_index(drop=True)

print("Shape original:", df.shape)
print("Shape filtrado:", df_filtered.shape)

df_filtered.head(10)

Shape original: (280, 19)
Shape filtrado: (126, 25)


,model,split,window_size,target,n_samples,balanced_accuracy_test,f1_macro_test,f1_weighted,accuracy,precision_macro,...,family,is_useful,is_strong,is_very_strong,balanced_accuracy_valid,f1_macro_valid,bal_acc_gain_vs_naive_valid,gain_gap_test_valid,is_consistent,is_very_consistent
0,gru_balanced,test,30,t2_dir_thr_90,93990,0.449329,0.449092,0.541916,0.540951,0.448880,...,gru_balanced,True,True,True,0.436482,0.436358,0.103149,0.012847,True,True
1,transformer_balanced,test,180,t2_dir_thr_120,64740,0.445395,0.445841,0.514053,0.519741,0.447254,...,transformer_balanced,True,True,True,0.409654,0.410077,0.076321,0.035741,False,False
2,gru_balanced,test,60,t2_dir_thr_90,88140,0.445165,0.438910,0.525453,0.530633,0.441109,...,gru_balanced,True,True,True,0.436680,0.440747,0.103347,0.008485,True,True
3,lstm_balanced,test,30,t2_dir_thr_90,93990,0.443933,0.444121,0.541269,0.544207,0.444887,...,lstm_balanced,True,True,True,0.430917,0.434321,0.097584,0.013016,True,True
4,gru_balanced,test,180,t2_dir_thr_120,64740,0.440567,0.406979,0.490302,0.524668,0.439066,...,gru_balanced,True,True,True,0.417997,0.412663,0.084664,0.022570,True,False
5,gru_balanced,test,30,t2_dir_thr_120,93990,0.440140,0.440524,0.539329,0.546473,0.442635,...,gru_balanced,True,True,True,0.414784,0.418992,0.081450,0.025357,True,False
6,xgboost_balanced,test,30,t2_dir_thr_90,93990,0.438932,0.438846,0.535633,0.537695,0.439061,...,xgboost_balanced,True,True,True,0.432357,0.435098,0.099024,0.006575,True,True
7,lstm_balanced,test,30,t2_dir_thr_120,93990,0.438271,0.435395,0.532906,0.536759,0.437192,...,lstm_balanced,True,True,True,0.421880,0.425907,0.088547,0.016390,True,True
8,lstm_balanced,test,60,t2_dir_thr_90,88140,0.438210,0.437117,0.528868,0.539800,0.440513,...,lstm_balanced,True,True,True,0.419434,0.423268,0.086101,0.018776,True,True
9,lightgbm_balanced,test,30,t2_dir_thr_90,93990,0.437927,0.437943,0.534988,0.536823,0.438141,...,lightgbm_balanced,True,True,True,0.429279,0.431511,0.095946,0.008648,True,True


In [46]:
print("Configs filtradas:", len(df_filtered))
print("Consistentes (gap <= 0.03):", df_filtered["is_consistent"].sum())
print("Muy consistentes (gap <= 0.02):", df_filtered["is_very_consistent"].sum())

Configs filtradas: 126
Consistentes (gap <= 0.03): 122
Muy consistentes (gap <= 0.02): 106


## **Observaciones**

a) Reducción del espacio de búsqueda

* Configuraciones originales: 280
* Configuraciones filtradas: 126

Esto implica:

* Se eliminó aproximadamente el 55% del espacio total
* El filtro removió configuraciones con señal débil o ruido
* Se mantiene un conjunto suficientemente amplio para análisis robusto

---

b) Alta proporción de configuraciones consistentes

* Consistentes (gap ≤ 0.03): 122 / 126
* Muy consistentes (gap ≤ 0.02): 106 / 126

Interpretación:

* La gran mayoría de configuraciones mantienen desempeño entre valid y test
* No se observa sobreajuste significativo a nivel general
* El dataset presenta buena estabilidad out-of-sample

---

c) Confirmación de señal robusta

* En las mejores configuraciones se observan gains elevados (≈ 0.10 – 0.11)
* Múltiples modelos alcanzan condiciones de señal fuerte (`is_very_strong = True`)

Interpretación:

* La señal detectada es real y no producto del azar
* Existen varias configuraciones capaces de capturar dicha señal

---

d) Modelos dominantes en el top

Se observa recurrencia de los siguientes modelos en las mejores posiciones:

* gru_balanced
* lstm_balanced
* transformer_balanced
* xgboost_balanced
* lightgbm_balanced

Interpretación:

* Los modelos no lineales dominan el problema
* Las arquitecturas más complejas capturan mejor la señal

---

e) Importancia del balanceo de clases

* Todas las configuraciones destacadas corresponden a modelos con `class_weight_mode = balanced`

Interpretación:

* El problema T2 presenta desbalance de clases relevante
* El uso de class_weight es crítico para capturar señal
* Modelos sin balanceo pierden capacidad predictiva

---

f) Gap reducido entre test y valid

Se observan diferencias típicas pequeñas:

* 0.008 – 0.02 en la mayoría de casos

Interpretación:

* Buena capacidad de generalización
* Estabilidad entre validación y test
* Indicio de pipeline correctamente diseñado (sin leakage evidente)

---

g) Casos puntuales de inconsistencia

* Algunas configuraciones presentan gaps más elevados (ej: ≈ 0.035)

Interpretación:

* Modelos más complejos pueden sobreajustar en ciertos casos
* No todos los resultados altos son necesariamente confiables
* Justifica el uso de métricas de consistencia

---

h) Evaluación global del filtro

* El filtro elimina configuraciones no robustas sin afectar las mejores
* Se logra un balance adecuado entre reducción de ruido y preservación de señal
* El dataset resultante es consistente y confiable

---

i) Implicación para el siguiente paso

* El conjunto filtrado permite realizar un ranking representativo
* Las decisiones posteriores se basarán en señal real y no en ruido
* Se encuentra preparado el dataset para la selección de modelos candidatos


# **4. Ranking por modelo (nivel arquitectura)**


Este paso tiene como objetivo evaluar el desempeño de cada arquitectura de modelo de forma agregada, utilizando únicamente configuraciones que ya demostraron ser válidas tras el filtro de calidad.

Al agrupar por modelo y promediar sus métricas, se obtiene una visión global de qué tipos de modelos capturan mejor la señal del problema, permitiendo comparar arquitecturas de manera justa y seleccionar las más prometedoras para el siguiente этап de tuning.

In [47]:
# Trabajar solo con datos filtrados (ya limpios)
df_rank = df_filtered.copy()

# Ranking por modelo usando TEST
ranking_models = (
    df_rank
    .groupby("model")
    .agg(
        gain_mean=("bal_acc_gain_vs_naive_test", "mean"),
        gain_median=("bal_acc_gain_vs_naive_test", "median"),
        gain_max=("bal_acc_gain_vs_naive_test", "max"),
        n_configs=("model", "count")
    )
    .sort_values(by="gain_mean", ascending=False)
)

ranking_models

,gain_mean,gain_median,gain_max,n_configs
model,,,,
gru_balanced,0.101022,0.101893,0.115995,10
lstm_balanced,0.096814,0.095637,0.110600,10
transformer_balanced,0.091691,0.089759,0.112062,10
lightgbm_balanced,0.090509,0.087605,0.104594,10
xgboost_balanced,0.090000,0.087018,0.105599,10
logistic_regression_balanced,0.080493,0.077359,0.095296,10
random_forest,0.019448,0.018928,0.023640,10
transformer,0.017553,0.014679,0.033360,5
gru,0.014983,0.015434,0.017584,8


## **Observaciones**

a) Dominio claro de modelos balanceados

* Los 6 mejores modelos son todos `*_balanced`
* Gains promedio ≈ **0.08 – 0.10**

Interpretación:

* El balanceo de clases es determinante
* Sin balanceo, los modelos pierden gran parte de la señal

---

b) Liderazgo de modelos secuenciales (deep learning)

Top 3:

* `gru_balanced` → **0.101**
* `lstm_balanced` → **0.096**
* `transformer_balanced` → **0.091**

Interpretación:

* Modelos que capturan dependencia temporal dominan
* El problema es claramente **secuencial (no tabular puro)**

---

c) Modelos de boosting muy competitivos

* `lightgbm_balanced` ≈ 0.090
* `xgboost_balanced` ≈ 0.090

Interpretación:

* Alternativa sólida a deep learning
* Capturan bien no linealidades sin modelar secuencia explícita

---

d) Logistic balanceado sorprendentemente fuerte

* `logistic_regression_balanced` ≈ **0.080**

Interpretación:

* Existe señal lineal relevante
* Buen baseline competitivo
* Útil como referencia en tuning

---

e) Caída drástica sin balanceo

Ejemplos:

* `transformer`: 0.017
* `gru`: 0.014
* `lstm`: 0.011

Interpretación:

* Sin class_weight → el modelo colapsa al baseline
* Confirma fuerte desbalance en T2

---

f) Random Forest débil

* `random_forest`: ≈ 0.019
* `random_forest_balanced`: ≈ 0.013

Interpretación:

* Bajo rendimiento relativo
* No competitivo frente a boosting o DL

---

g) Consistencia de resultados

* Todos los modelos top tienen `n_configs = 10`
* Mediana ≈ media

Interpretación:

* No son outliers
* Performance estable dentro del modelo

---

h) Conclusión operativa

Hay 3 grupos claros:

1. **Top (0.09–0.10)** → GRU, LSTM, Transformer
2. **Muy buenos (≈0.09)** → LightGBM, XGBoost
3. **Aceptable (≈0.08)** → Logistic balanceado



# **5. Análisis por window_size (L)**


Este paso tiene como objetivo evaluar cómo influye el tamaño de la ventana temporal (window_size) en la capacidad predictiva de los modelos.

Al agrupar los resultados por ventana y analizar el rendimiento promedio, se busca identificar qué escala temporal captura mejor la señal del mercado, permitiendo distinguir si ventanas más cortas están dominadas por ruido o si ventanas más largas logran representar patrones más estables y predecibles.

In [48]:
ranking_window = (
    df_filtered
    .groupby("window_size")
    .agg(
        gain_mean=("bal_acc_gain_vs_naive_test", "mean"),
        gain_median=("bal_acc_gain_vs_naive_test", "median"),
        gain_max=("bal_acc_gain_vs_naive_test", "max"),
        n_configs=("window_size", "count")
    )
    .sort_values(by="gain_mean", ascending=False)
)

ranking_window

,gain_mean,gain_median,gain_max,n_configs
window_size,,,,
30,0.061354,0.074308,0.115995,22
60,0.050791,0.020936,0.111832,26
180,0.049367,0.017606,0.112062,27
120,0.048061,0.020659,0.091788,25
90,0.047905,0.019201,0.102273,26


## **Observaciones**

* `gain_mean` → desempeño promedio
* `gain_median` → robustez
* `gain_max` → potencial
* `n_configs` → cuántas configs sobrevivieron

---

a) La mejor ventana en promedio es L = 30

* `gain_mean`: **0.0613** (la más alta)
* `gain_median`: **0.0743** (también la más alta)

Interpretación:

* Es la única ventana donde **media ≈ mediana alta**
* Indica señal **consistente y robusta**, no dependiente de outliers

---

b) Ventanas largas tienen menor consistencia

Ejemplo L = 180:

* mean: 0.0493
* median: 0.0176

Interpretación:

* gran diferencia → señal concentrada en pocos casos
* mayoría de configuraciones → señal débil

Esto se repite en:

* L = 60
* L = 90
* L = 120

---

c) Todas las ventanas tienen potencial (gain_max alto)

* Todas alcanzan ≈ **0.10 – 0.11**

Interpretación:

* la señal existe en todas las escalas
* pero no todas la capturan de forma estable

---

d) Trade-off claro: consistencia vs especialización

* L = 30 →

  * más consistente
  * más generalizable

* L ≥ 60 →

  * menos consistente
  * pero con casos puntuales muy buenos

---

e) Cantidad de configuraciones equilibrada

* n_configs ≈ 22–27 en todas

Interpretación:

* comparación justa
* no hay sesgo por cantidad

---

f) Conclusión técnica

* L = 30 es la ventana más robusta
* ventanas largas no son malas, pero:

  * dependen más del modelo
  * tienen mayor varianza

---

g) Implicación para el siguiente paso

* L = 30 debe estar sí o sí en la selección final
* no descartar L = 60 o 180 todavía
* decidir completamente después del análisis conjunto (modelo + target)



# **6. Análisis por target (90 vs 120)**

Este paso busca evaluar qué horizonte (target) es más predecible, comparando el desempeño promedio de los modelos entre t2_dir_thr_90 y t2_dir_thr_120.

El objetivo es identificar cuál de los dos presenta mayor separabilidad de clases y, por lo tanto, mayor señal explotable.

In [49]:
ranking_target = (
    df_filtered
    .groupby("target")
    .agg(
        gain_mean=("bal_acc_gain_vs_naive_test", "mean"),
        gain_median=("bal_acc_gain_vs_naive_test", "median"),
        gain_max=("bal_acc_gain_vs_naive_test", "max"),
        n_configs=("target", "count")
    )
    .sort_values(by="gain_mean", ascending=False)
)

ranking_target

,gain_mean,gain_median,gain_max,n_configs
target,,,,
t2_dir_thr_120,0.051349,0.026492,0.112062,62
t2_dir_thr_90,0.051042,0.022212,0.115995,64


## **Observaciones**

* `gain_mean` → qué target es mejor en promedio
* `gain_median` → cuál es más robusto
* `gain_max` → potencial máximo
* `n_configs` → equilibrio de muestra

**Qué esperamos responder**

* ¿El horizonte corto (90) captura mejor microestructura?
* ¿El horizonte más largo (120) reduce ruido y mejora señal?


**Las observaciones claras para documentar:**

---

a) Desempeño promedio prácticamente idéntico

* `t2_dir_thr_120`: gain_mean ≈ **0.05135**
* `t2_dir_thr_90`: gain_mean ≈ **0.05104**

Interpretación:

* Ambos targets tienen **el mismo nivel de señal en promedio**
* No hay ventaja clara en términos de performance global

---

b) Ligera ventaja de robustez en 120

* median (120): **0.0265**
* median (90): **0.0222**

Interpretación:

* El target 120 tiene una distribución ligeramente más estable
* Menos dependencia de outliers

---

c) Mayor potencial máximo en 90

* max (90): **0.11599**
* max (120): **0.11206**

Interpretación:

* El target 90 alcanza configuraciones ligeramente más extremas
* Puede capturar mejor ciertos patrones puntuales

---

d) Balance perfecto de configuraciones

* 62 vs 64 configuraciones

Interpretación:

* Comparación completamente justa
* No hay sesgo en la muestra

---

e) Conclusión técnica

* Ambos targets son **igualmente válidos**
* No hay evidencia para descartar ninguno
* Diferencia clave:

  * `90` → mayor potencial (más agresivo)
  * `120` → mayor estabilidad (más robusto)

---

f) Implicación para el proceso

* Se deben mantener **ambos targets** en el análisis siguiente
* La decisión final no se toma aquí
* Se definirá en el análisis conjunto (modelo + ventana + target)

---

g) Insight importante

Esto confirma algo relevante del problema:

* la señal no depende fuertemente del horizonte
* depende más de:

  * modelo
  * window_size
  * interacción entre variables

# **7. Análisis conjunto (modelo + L + target)**


Este paso tiene como objetivo identificar las mejores configuraciones reales considerando la interacción conjunta entre modelo, tamaño de ventana (window_size) y target.

A diferencia de los análisis anteriores, que evaluaban cada dimensión por separado, aquí se analizan combinaciones específicas para detectar qué setups concretos logran capturar mejor la señal.

El resultado es un ranking de las mejores configuraciones, que servirá como base directa para la selección final de candidatos a tunear.

In [50]:
# ================================
# 7. Análisis conjunto
# model + window_size + target
# ================================

ranking_joint = (
    df_filtered
    .groupby(["model", "window_size", "target"])
    .agg(
        gain_mean=("bal_acc_gain_vs_naive_test", "mean"),
        gain_median=("bal_acc_gain_vs_naive_test", "median"),
        gain_max=("bal_acc_gain_vs_naive_test", "max"),
        bal_acc_mean=("balanced_accuracy_test", "mean"),
        f1_macro_mean=("f1_macro_test", "mean"),
        gain_valid_mean=("bal_acc_gain_vs_naive_valid", "mean"),
        gap_mean=("gain_gap_test_valid", "mean"),
        n_configs=("model", "count")
    )
    .sort_values(by=["gain_mean", "gain_median"], ascending=False)
    .reset_index()
)

ranking_joint.head(20)

,model,window_size,target,gain_mean,gain_median,gain_max,bal_acc_mean,f1_macro_mean,gain_valid_mean,gap_mean,n_configs
0,gru_balanced,30,t2_dir_thr_90,0.115995,0.115995,0.115995,0.449329,0.449092,0.103149,0.012847,1
1,transformer_balanced,180,t2_dir_thr_120,0.112062,0.112062,0.112062,0.445395,0.445841,0.076321,0.035741,1
2,gru_balanced,60,t2_dir_thr_90,0.111832,0.111832,0.111832,0.445165,0.438910,0.103347,0.008485,1
3,lstm_balanced,30,t2_dir_thr_90,0.110600,0.110600,0.110600,0.443933,0.444121,0.097584,0.013016,1
4,gru_balanced,180,t2_dir_thr_120,0.107234,0.107234,0.107234,0.440567,0.406979,0.084664,0.022570,1
5,gru_balanced,30,t2_dir_thr_120,0.106807,0.106807,0.106807,0.440140,0.440524,0.081450,0.025357,1
6,xgboost_balanced,30,t2_dir_thr_90,0.105599,0.105599,0.105599,0.438932,0.438846,0.099024,0.006575,1
7,lstm_balanced,30,t2_dir_thr_120,0.104937,0.104937,0.104937,0.438271,0.435395,0.088547,0.016390,1
8,lstm_balanced,60,t2_dir_thr_90,0.104877,0.104877,0.104877,0.438210,0.437117,0.086101,0.018776,1
9,lightgbm_balanced,30,t2_dir_thr_90,0.104594,0.104594,0.104594,0.437927,0.437943,0.095946,0.008648,1


In [51]:
top_joint_consistent = (
    ranking_joint[ranking_joint["gap_mean"] <= 0.03]
    .sort_values(by=["gain_mean", "gain_median"], ascending=False)
    .reset_index(drop=True)
)

top_joint_consistent.head(20)

,model,window_size,target,gain_mean,gain_median,gain_max,bal_acc_mean,f1_macro_mean,gain_valid_mean,gap_mean,n_configs
0,gru_balanced,30,t2_dir_thr_90,0.115995,0.115995,0.115995,0.449329,0.449092,0.103149,0.012847,1
1,gru_balanced,60,t2_dir_thr_90,0.111832,0.111832,0.111832,0.445165,0.438910,0.103347,0.008485,1
2,lstm_balanced,30,t2_dir_thr_90,0.110600,0.110600,0.110600,0.443933,0.444121,0.097584,0.013016,1
3,gru_balanced,180,t2_dir_thr_120,0.107234,0.107234,0.107234,0.440567,0.406979,0.084664,0.022570,1
4,gru_balanced,30,t2_dir_thr_120,0.106807,0.106807,0.106807,0.440140,0.440524,0.081450,0.025357,1
5,xgboost_balanced,30,t2_dir_thr_90,0.105599,0.105599,0.105599,0.438932,0.438846,0.099024,0.006575,1
6,lstm_balanced,30,t2_dir_thr_120,0.104937,0.104937,0.104937,0.438271,0.435395,0.088547,0.016390,1
7,lstm_balanced,60,t2_dir_thr_90,0.104877,0.104877,0.104877,0.438210,0.437117,0.086101,0.018776,1
8,lightgbm_balanced,30,t2_dir_thr_90,0.104594,0.104594,0.104594,0.437927,0.437943,0.095946,0.008648,1
9,lightgbm_balanced,30,t2_dir_thr_120,0.103971,0.103971,0.103971,0.437305,0.437990,0.079242,0.024729,1


## **Observaciones**

a) Las mejores configuraciones están altamente concentradas

* Todas las combinaciones top tienen:

  * gain ≈ **0.10 – 0.116**
* No hay dispersión grande en el top

Interpretación:

* Existe un “techo” claro de performance
* Varias configuraciones alcanzan ese nivel → señal sólida

---

b) Dominio claro de ciertos modelos

Repetición fuerte en el top:

* `gru_balanced` (dominante absoluto)
* `lstm_balanced`
* `xgboost_balanced`
* `lightgbm_balanced`

Interpretación:

* GRU es el modelo más consistente en top
* LSTM también muy competitivo
* Boosting compite bien con deep learning

---

c) Transformer pierde protagonismo

* Aparece menos en el top consistente
* Algunos casos tienen mayor gap

Interpretación:

* Más sensible a overfitting
* No es el mejor candidato principal

---

d) Confirmación fuerte de L = 30

* La mayoría del top usa **window_size = 30**

Ejemplos:

* GRU (30, 90) → #1
* LSTM (30, 90) → top
* XGBoost (30, 90) → top
* LightGBM (30, 90) → top

Interpretación:

* L = 30 captura mejor la señal
* Ventanas largas aparecen, pero menos dominantes

---

e) Target dominante: 90

* `t2_dir_thr_90` aparece más frecuentemente en el top

Interpretación:

* Horizonte corto captura mejor microestructura
* Más señal explotable

---

f) Consistencia valid–test

* La mayoría tiene:

  * gap bajo (< 0.02)
* Incluso algunos:

  * gap ≈ 0

Interpretación:

* Configuraciones muy robustas
* Excelente generalización

---

g) Configuraciones “estrella” claras

Ejemplo top absoluto:

* `gru_balanced | L=30 | t2_dir_thr_90`

  * gain ≈ **0.116**
  * gap bajo

👉 Esta es una referencia clave del problema

---

h) Conclusión técnica

El patrón es claro:

* Modelo → **GRU / LSTM dominan**
* Ventana → **30 domina**
* Target → **90 ligeramente mejor**

---

i) Implicación directa

Ya tienes lo necesario para el punto 8:

* modelos candidatos
* ventanas candidatas
* targets candidatos


#**8. Selección final de candidatos**


a) Selección de modelos

Se seleccionan tres modelos representativos basados en el análisis previo:

* gru_balanced

  * Modelo con mejor desempeño global
  * Mayor presencia en las configuraciones top
  * Alta consistencia entre validación y test

* xgboost_balanced

  * Modelo ensemble con rendimiento competitivo frente a deep learning
  * Buena estabilidad y capacidad de generalización

* logistic_regression_balanced

  * Modelo baseline con desempeño aceptable
  * Permite interpretar la señal y medir mejoras reales

---

b) Selección de ventanas (window_size)

Se seleccionan tres escalas temporales:

* 30

  * Mayor desempeño promedio y mediano
  * Alta consistencia
  * Dominante en las mejores configuraciones

* 60

  * Buen equilibrio entre desempeño y estabilidad
  * Presente en múltiples configuraciones relevantes

* 180

  * Menor consistencia, pero alto potencial en configuraciones específicas
  * Permite capturar dinámicas de más largo plazo

---

c) Selección de targets

Se mantienen ambos horizontes:

* t2_dir_thr_90

  * Mayor presencia en configuraciones top
  * Captura mejor patrones de corto plazo

* t2_dir_thr_120

  * Ligera mayor estabilidad en la distribución
  * Complementa el horizonte más corto

---

d) Espacio final de búsqueda

La combinación seleccionada define el espacio de tuning:

* 3 modelos
* 3 ventanas
* 2 targets

Total:

* 18 configuraciones

---

e) Conclusión general

La selección se basa exclusivamente en evidencia empírica obtenida del análisis previo.

Se logra un equilibrio entre:

* modelos con mayor capacidad predictiva
* diversidad de enfoques (deep learning, ensemble, baseline)
* cobertura de distintas escalas temporales
* estabilidad entre horizontes de predicción

El espacio resultante es lo suficientemente reducido para un tuning eficiente, sin comprometer la capacidad de capturar la señal presente en los datos.


# **9. Definición del espacio de tuning**


a) Objetivo

Definir un espacio de hiperparámetros acotado para cada modelo seleccionado, permitiendo realizar un proceso de tuning eficiente sin explorar combinaciones innecesarias.

---

b) Principios de diseño

* Limitar el número de hiperparámetros a los más influyentes
* Definir rangos realistas (basados en experiencia y resultados previos)
* Evitar combinaciones extremas que generen inestabilidad
* Mantener comparabilidad entre modelos

---

c) Modelo 1: gru_balanced

Hiperparámetros clave:

* hidden_size
* num_layers
* dropout
* learning_rate
* batch_size

Espacio sugerido:

```python
gru_params = {
    "hidden_size": [64, 128, 256],
    "num_layers": [1, 2],
    "dropout": [0.1, 0.2, 0.3],
    "learning_rate": [1e-3, 5e-4, 1e-4],
    "batch_size": [1024, 2048]
}
```

---

d) Modelo 2: xgboost_balanced

Hiperparámetros clave:

* n_estimators
* max_depth
* learning_rate
* subsample
* colsample_bytree
* reg_lambda

Espacio sugerido:

```python
xgb_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.1, 0.05, 0.01],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_lambda": [1, 5, 10]
}
```

---

e) Modelo 3: logistic_regression_balanced

Hiperparámetros clave:

* C (regularización)
* penalty
* solver

Espacio sugerido:

```python
logreg_params = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "solver": ["lbfgs"]
}
```

---

f) Tamaño del espacio de búsqueda

* GRU: 3 × 2 × 3 × 3 × 2 = 108 combinaciones
* XGBoost: 3 × 3 × 3 × 2 × 2 × 3 = 324 combinaciones
* Logistic: 4 combinaciones

Interpretación:

* Espacio manejable
* Permite usar grid search o random search
* Balance entre exploración y costo computacional

---

g) Conclusión

El espacio de tuning definido:

* se enfoca en los hiperparámetros más relevantes
* evita combinaciones innecesarias
* mantiene un tamaño controlado
* permite optimizar cada modelo de forma eficiente y comparable

---

Si quieres, el siguiente paso es armar el **pipeline de tuning unificado** para los tres modelos.
